### 🧹 AWS Cleanup Script: Scanned & Deleted Resources

The cleanup script targets **9 core AWS services** across both global and region-specific infrastructure, prioritizing components that incur ongoing hourly or storage charges.

| Service | Specific Resource Scanned | What Gets Deleted |
| :--- | :--- | :--- |
| **S3** *(Global)* | Buckets | Empties all objects, version histories, and delete markers, then deletes the bucket itself. |
| **SageMaker** | Real-time Endpoints | Active inference endpoints incurring hourly compute charges. |
| **SageMaker** | Endpoint Configurations | Saved deployment configurations for model hosting. |
| **SageMaker** | Models | Created SageMaker model entities and container definitions. |
| **EKS** | Managed Kubernetes Clusters | The control plane for EKS clusters. |
| **CloudFormation** | Active Stacks | Automatically updates termination protection to `False` and deletes stack resources (with a `RetainResources` fallback if trapped in `DELETE_FAILED`). |
| **ECR** | Container Repositories | Repositories holding Docker images (forces deletion even if images exist). |
| **Elastic Load Balancing** | Application / Network Load Balancers | Active load balancers (including those auto-provisioned by EKS services). |
| **EC2 / Storage** | Unattached EBS Volumes | Idle block storage volumes sitting in the `available` state. |
| **EC2 / Networking** | Unattached Elastic IPs (EIPs) | Public IPv4 addresses not associated with a running EC2 instance or Load Balancer. |

In [1]:
!pip install boto3
import os
import boto3
from botocore.exceptions import BotoCoreError, ClientError

# Set DRY_RUN = False to perform actual deletion!
DRY_RUN = False

def get_colab_secret(key_name: str, required: bool = True):
    """Safely retrieves a secret from Google Colab secrets if available."""
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except Exception:
        if required:
            print(f"⚠️ Warning: Could not retrieve {key_name} from Colab secrets.")
        return None

def get_all_enabled_regions(session=None) -> list:
    """Dynamically retrieves all enabled EC2 regions for the AWS account."""
    try:
        ec2_client = (session or boto3).client("ec2", region_name="us-east-1")
        response = ec2_client.describe_regions(AllRegions=False)
        return [region["RegionName"] for region in response["Regions"]]
    except Exception as exc:
        print(f"⚠️ Could not fetch regions dynamically ({exc}). Falling back to defaults.")
        return ["eu-north-1", "us-east-1", "us-west-2", "eu-west-1"]

def get_failed_resource_ids(cfn_client, stack_name: str) -> list:
    """Retrieves logical resource IDs trapped in a DELETE_FAILED state."""
    failed_ids = []
    try:
        paginator = cfn_client.get_paginator("list_stack_resources")
        for page in paginator.paginate(StackName=stack_name):
            for resource in page.get("StackResourceSummaries", []):
                if resource.get("ResourceStatus") == "DELETE_FAILED":
                    failed_ids.append(resource["LogicalResourceId"])
        return failed_ids
    except Exception:
        return []

def empty_and_delete_s3_bucket(s3_resource, bucket_name: str, dry_run: bool = True):
    """Empties all objects, versions, and delete markers from an S3 bucket and deletes it."""
    try:
        bucket = s3_resource.Bucket(bucket_name)

        if not dry_run:
            # 1. Purge all object versions and delete markers
            bucket.object_versions.delete()
            # 2. Delete the empty bucket
            bucket.delete()
            print(f"    ✅ Successfully emptied and deleted S3 Bucket: {bucket_name}")
        else:
            print(f"    🛡️ [DRY-RUN] Would purge contents and delete S3 Bucket: {bucket_name}")
    except Exception as exc:
        print(f"    ❌ Error deleting S3 Bucket {bucket_name}: {exc}")

def nuke_all_aws_resources(dry_run: bool = True):
    """
    Scans and deletes ALL active billing resources across SageMaker, EKS,
    CloudFormation, ECR, Load Balancers, EBS, EIPs, and S3 Buckets in ALL enabled regions.
    """
    if dry_run:
        print("🛡️  RUNNING IN DRY-RUN MODE: No resources will be deleted.\n")
    else:
        print("💥 CRITICAL WARNING: DRY-RUN IS DISABLED. Wiping ALL detected resources!\n")

    access_key = os.environ.get("AWS_ACCESS_KEY_ID") or get_colab_secret("AWS_ACCESS_KEY_ID")
    secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY") or get_colab_secret("AWS_SECRET_ACCESS_KEY")
    session_token = os.environ.get("AWS_SESSION_TOKEN") or get_colab_secret("AWS_SESSION_TOKEN", required=False)

    init_session = boto3.session.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        aws_session_token=session_token,
        region_name="us-east-1",
    )

    regions = get_all_enabled_regions(session=init_session)
    total_found = 0

    # ==========================================
    # GLOBAL RESOURCE CLEANUP: S3 BUCKETS
    # ==========================================
    print("==========================================")
    print("Scanning Global Service: S3 Buckets")
    print("==========================================")
    try:
        s3_client = init_session.client("s3")
        s3_resource = init_session.resource("s3")
        buckets = s3_client.list_buckets().get("Buckets", [])

        if not buckets:
            print("  ✅ No S3 Buckets found.")
        else:
            for b in buckets:
                name = b["Name"]
                print(f"  ❌ Found S3 Bucket: {name}")
                total_found += 1
                empty_and_delete_s3_bucket(s3_resource, name, dry_run=dry_run)
    except Exception as exc:
        print(f"  Error checking S3 Buckets: {exc}")

    # ==========================================
    # REGION-SPECIFIC RESOURCE CLEANUP
    # ==========================================
    for region_name in regions:
        print(f"\n==========================================")
        print(f"Scanning Region: {region_name}")
        print(f"==========================================")
        try:
            session = boto3.session.Session(
                region_name=region_name,
                aws_access_key_id=access_key,
                aws_secret_access_key=secret_key,
                aws_session_token=session_token,
            )
            sm = session.client("sagemaker", region_name=region_name)
            eks = session.client("eks", region_name=region_name)
            cfn = session.client("cloudformation", region_name=region_name)
            ecr = session.client("ecr", region_name=region_name)
            ec2 = session.client("ec2", region_name=region_name)
            elbv2 = session.client("elbv2", region_name=region_name)

            # 1. SAGEMAKER ENDPOINTS
            try:
                for page in sm.get_paginator("list_endpoints").paginate():
                    for ep in page.get("Endpoints", []):
                        name = ep["EndpointName"]
                        print(f"  ❌ [{region_name}] SageMaker Endpoint: {name}")
                        total_found += 1
                        if not dry_run:
                            sm.delete_endpoint(EndpointName=name)
            except Exception as e:
                print(f"  Error checking Endpoints: {e}")

            # 2. SAGEMAKER ENDPOINT CONFIGS
            try:
                for page in sm.get_paginator("list_endpoint_configs").paginate():
                    for cfg in page.get("EndpointConfigs", []):
                        name = cfg["EndpointConfigName"]
                        print(f"  ❌ [{region_name}] SageMaker EndpointConfig: {name}")
                        total_found += 1
                        if not dry_run:
                            sm.delete_endpoint_config(EndpointConfigName=name)
            except Exception as e:
                print(f"  Error checking Endpoint Configs: {e}")

            # 3. SAGEMAKER MODELS
            try:
                for page in sm.get_paginator("list_models").paginate():
                    for model in page.get("Models", []):
                        name = model["ModelName"]
                        print(f"  ❌ [{region_name}] SageMaker Model: {name}")
                        total_found += 1
                        if not dry_run:
                            sm.delete_model(ModelName=name)
            except Exception as e:
                print(f"  Error checking Models: {e}")

            # 4. EKS CLUSTERS
            try:
                clusters = eks.list_clusters().get("clusters", [])
                for cluster_name in clusters:
                    print(f"  ❌ [{region_name}] EKS Cluster: {cluster_name}")
                    total_found += 1
                    if not dry_run:
                        eks.delete_cluster(name=cluster_name)
            except Exception as e:
                print(f"  Error checking EKS: {e}")

            # 5. CLOUDFORMATION STACKS
            try:
                statuses = ["CREATE_COMPLETE", "UPDATE_COMPLETE", "ROLLBACK_COMPLETE", "DELETE_FAILED"]
                for page in cfn.get_paginator("list_stacks").paginate(StackStatusFilter=statuses):
                    for stack in page.get("StackSummaries", []):
                        name = stack["StackName"]
                        print(f"  ❌ [{region_name}] CloudFormation Stack: {name}")
                        total_found += 1
                        if not dry_run:
                            cfn.update_termination_protection(EnableTerminationProtection=False, StackName=name)
                            failed_ids = get_failed_resource_ids(cfn, name)
                            if failed_ids:
                                cfn.delete_stack(StackName=name, RetainResources=failed_ids)
                            else:
                                cfn.delete_stack(StackName=name)
            except Exception as e:
                print(f"  Error checking Stacks: {e}")

            # 6. ECR REPOSITORIES
            try:
                repos = ecr.describe_repositories().get("repositories", [])
                for repo in repos:
                    name = repo["repositoryName"]
                    print(f"  ❌ [{region_name}] ECR Repo: {name}")
                    total_found += 1
                    if not dry_run:
                        ecr.delete_repository(repositoryName=name, force=True)
            except Exception as e:
                print(f"  Error checking ECR: {e}")

            # 7. ELASTIC LOAD BALANCERS
            try:
                lbs = elbv2.describe_load_balancers().get("LoadBalancers", [])
                for lb in lbs:
                    arn = lb["LoadBalancerArn"]
                    name = lb["LoadBalancerName"]
                    print(f"  ❌ [{region_name}] Elastic Load Balancer: {name}")
                    total_found += 1
                    if not dry_run:
                        elbv2.delete_load_balancer(LoadBalancerArn=arn)
            except Exception as e:
                print(f"  Error checking Load Balancers: {e}")

            # 8. UNATTACHED EBS VOLUMES
            try:
                volumes = ec2.describe_volumes(Filters=[{"Name": "status", "Values": ["available"]}]).get("Volumes", [])
                for vol in volumes:
                    vol_id = vol["VolumeId"]
                    print(f"  ❌ [{region_name}] Unattached EBS Volume: {vol_id}")
                    total_found += 1
                    if not dry_run:
                        ec2.delete_volume(VolumeId=vol_id)
            except Exception as e:
                print(f"  Error checking Volumes: {e}")

            # 9. UNATTACHED EIPs
            try:
                eips = ec2.describe_addresses().get("Addresses", [])
                for eip in eips:
                    if "AssociationId" not in eip:
                        alloc_id = eip["AllocationId"]
                        print(f"  ❌ [{region_name}] Unattached Elastic IP: {eip.get('PublicIp')}")
                        total_found += 1
                        if not dry_run:
                            ec2.release_address(AllocationId=alloc_id)
            except Exception as e:
                print(f"  Error checking Elastic IPs: {e}")

        except Exception as exc:
            print(f"[{region_name}] Access failed: {exc}")

    print(f"\n==========================================")
    print(f"SWEEP COMPLETE: Found {total_found} total active resource(s) across S3 and {len(regions)} region(s).")
    if dry_run:
        print("Set `DRY_RUN = False` to delete all detected resources.")
    print(f"==========================================")

# Execute in Dry-Run Mode first
nuke_all_aws_resources(dry_run=DRY_RUN)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.0 MB/s eta 0:00:00
💥 CRITICAL WARNING: DRY-RUN IS DISABLED. Wiping ALL detected resources!

Scanning Global Service: S3 Buckets
  ❌ Found S3 Bucket: usecase-etl-1-455865672536
    ✅ Successfully emptied and deleted S3 Bucket: usecase-etl-1-455865672536
  ❌ Found S3 Bucket: usecase-etl-2-455865672536
    ✅ Successfully emptied and deleted S3 Bucket: usecase-etl-2-455865672536

Scanning Region: ap-south-1

Scanning Region: eu-north-1

Scanning Region: eu-west-3

Scanning Region: eu-west-2

Scanning Region: eu-west-1

Scanning Region: ap-northeast-3

Scanning Region: ap-northeast-2

Scanning Region: ap-northeast-1

Scanning Region: ca-central-1

Scanning Region: sa-east-1

Scanning Region: ap-southeast-1

Scanning Region: ap-southeast-2

Scanning Region: eu-central-1

Scanni

In [2]:
import os
import boto3

def get_bucket_region(s3_client, bucket_name: str) -> str:
    """Detect the exact AWS region where an S3 bucket resides."""
    try:
        location = s3_client.get_bucket_location(Bucket=bucket_name).get("LocationConstraint")
        # AWS returns None for us-east-1 and "EU" for legacy eu-west-1
        if location is None:
            return "us-east-1"
        if location == "EU":
            return "eu-west-1"
        return location
    except Exception:
        return "us-east-1"

def delete_usecase_etl_buckets(dry_run: bool = True):
    try:
        access_key = os.environ.get("AWS_ACCESS_KEY_ID") or get_colab_secret("AWS_ACCESS_KEY_ID")
        secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY") or get_colab_secret("AWS_SECRET_ACCESS_KEY")
        session_token = os.environ.get("AWS_SESSION_TOKEN") or get_colab_secret("AWS_SESSION_TOKEN", required=False)

        session = boto3.session.Session(
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            aws_session_token=session_token
        )
    except Exception as exc:
        print(f"❌ Could not build AWS session: {exc}")
        return

    # Global client used to scan account buckets across all regions
    global_s3_client = session.client("s3")

    try:
        account_id = session.client("sts").get_caller_identity()["Account"]
    except Exception as exc:
        print(f"❌ Could not resolve AWS account id: {exc}")
        return

    expected = {f"usecase-etl-{i}-{account_id}" for i in (1, 2)}
    buckets = global_s3_client.list_buckets().get("Buckets", [])
    targets = [b["Name"] for b in buckets if b["Name"] in expected or b["Name"].startswith("usecase-etl-")]

    print(f"🗂️ Found {len(targets)} UseCase ETL bucket(s) to remove across all regions.")

    for name in targets:
        # Dynamically resolve region for each bucket before deletion operations
        bucket_region = get_bucket_region(global_s3_client, name)
        print(f"📍 Target '{name}' resides in region: {bucket_region}")

        # Instantiate region-specific resource to execute deletion cleanly
        regional_s3_resource = session.resource("s3", region_name=bucket_region)
        empty_and_delete_s3_bucket(regional_s3_resource, name, dry_run=dry_run)

    print(f"✅ UseCase ETL bucket cleanup complete ({len(targets)} bucket(s) handled).")

delete_usecase_etl_buckets(dry_run=DRY_RUN)

🗂️ Found 0 UseCase ETL bucket(s) to remove across all regions.
✅ UseCase ETL bucket cleanup complete (0 bucket(s) handled).


In [3]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-14 02:10:21
